# Job Hunting: how UMBC computing grads find their first job, and what happens next
Alumni only. First-job facts come from `alumni`; everything after the first job comes from `employment_history`.

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine
from scipy import stats

# same data source as file.ipynb: Tiger Data when TIGER_URL is in .env, otherwise the local CSVs
load_dotenv()
tigerUrl = os.getenv("TIGER_URL")
engine = create_engine(tigerUrl.replace("postgres://", "postgresql+psycopg://", 1)) if tigerUrl else None

def load(table):
    if engine is None:
        return pd.read_csv(f"data/{table}.csv", na_values="Not Applicable")
    return pd.read_sql(f"SELECT * FROM {table} ORDER BY 1", engine)

stuAlum = load("alumni")
jobs = load("employment_history")
jobs["start_date"] = pd.to_datetime(jobs["start_date"])
hired = stuAlum[stuAlum["first_employer"].notna()]   # alumni who took a first job

In [ ]:
# Chart: what alumni were doing ~6 months after graduation, by graduation year, 100% stacked area
# "No Response" left out (unknown, not unemployed); 2026 grads were surveyed soon after graduating, so "still seeking" runs high
groups = {"Employed Full-Time": "Employed full-time", "Employed Part-Time": "Part-time or military",
          "Military": "Part-time or military", "Continuing Education": "Grad school", "Still Seeking": "Still seeking"}
colors = {"Employed full-time": "#2a78d6", "Part-time or military": "#eda100", "Grad school": "#1baf7a", "Still seeking": "#eb6834"}
responded = stuAlum[stuAlum["first_destination"] != "No Response"]
mix = pd.crosstab(responded["graduation_year"], responded["first_destination"].map(groups), normalize="index")[list(colors)] * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.stackplot(mix.index, mix.T, colors=list(colors.values()), labels=list(colors), edgecolor="white", linewidth=1.5)
middle = mix.cumsum(axis=1) - mix / 2
for name in colors:
    ax.annotate(f"{name} {mix[name].iloc[-1]:.0f}%", (mix.index[-1], middle[name].iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", color="#52514e", fontsize=9)
ax.set(xlim=(mix.index[0], mix.index[-1] + 3.2), ylim=(0, 100), xticks=mix.index[::2],
       ylabel="% of alumni who responded", title="Most grads are working 6 months out; 2020 was the hardest year")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.08), ncol=4, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Chart: how fast alumni start their first job, cumulative step chart (alumni with a first job)
months = np.sort(hired["months_to_first_job"].dropna())
pctHired = np.arange(1, len(months) + 1) / len(months) * 100
atGraduation = (months <= 0).mean() * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.step(months, pctHired, where="post", color="#2a78d6", linewidth=2)
ax.annotate(f"{atGraduation:.0f}% start right at graduation", (0, atGraduation), xytext=(10, 0),
            textcoords="offset points", va="center", color="#52514e", fontsize=9)
for q in (0.5, 0.75, 0.9):
    v = np.quantile(months, q)
    ax.plot(v, q * 100, "o", color="#2a78d6", markersize=8)
    ax.annotate(f"{q:.0%} started by {v:.1f} months", (v, q * 100), xytext=(10, -2), textcoords="offset points",
                va="top", color="#52514e", fontsize=9)
ax.set(xlim=(0, months.max()), ylim=(0, 102), xlabel="Months from graduation to first job",
       ylabel="% of hired alumni", title="Half of grads start their first job within a month")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Chart: how alumni found their first job, share of first jobs with median months to start, bar chart (return offers highlighted)
byChannel = (hired.groupby("first_job_found_via")
             .agg(share=("campus_id", "size"), months=("months_to_first_job", "median"))
             .sort_values("share"))
byChannel["share"] = byChannel["share"] / len(hired) * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(byChannel.index, byChannel["share"], height=0.7,
        color=["#2a78d6" if c == "Return Offer from Internship" else "#c3c2b7" for c in byChannel.index])
ax.bar_label(ax.containers[0], labels=[f"{s:.0f}%  ·  {m:.1f} months" for s, m in zip(byChannel["share"], byChannel["months"])],
             padding=3, color="#52514e")
ax.margins(x=0.25)
ax.set(xlabel="% of alumni first jobs  (label: share · median months to start)",
       title="Return offers are the most common route and by far the fastest")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Chart: industries of alumni first jobs, column chart with the federal ecosystem highlighted
federal = ["Federal Contracting", "Federal Government", "Defense & Aerospace"]
share = hired["first_employer_industry"].value_counts(normalize=True) * 100
share = pd.concat([share.head(9), pd.Series({"Other": share.iloc[9:].sum()})])

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar([textwrap.fill(i, 10, break_long_words=False) for i in share.index], share, width=0.7,
       color=["#2a78d6" if i in federal else "#c3c2b7" for i in share.index])
ax.bar_label(ax.containers[0], fmt="%.0f%%", padding=3, color="#52514e")
ax.tick_params(axis="x", labelsize=8)
ax.set(ylabel="% of alumni first jobs",
       title=f"{share[federal].sum():.0f}% of first jobs are in the federal ecosystem (blue)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Chart: median first-job salary by region, paycheck vs adjusted for cost of living, dumbbell (regions with >= 100 first jobs)
# adjusted = salary * 100 / cost_of_living_index (100 = national average; Baltimore is exactly 100)
# only gaps that pass Mann-Whitney with Bonferroni correction are claimed in the title (checks table at the end)
firstJobs = jobs[jobs["change_type"] == "First Job"]
firstJobs = firstJobs.assign(adjusted=firstJobs["annual_salary_usd"] * 100 / firstJobs["cost_of_living_index"])
byRegion = firstJobs.groupby("region").agg(n=("job_id", "size"), paycheck=("annual_salary_usd", "median"),
                                           adjusted=("adjusted", "median"))
byRegion = byRegion[byRegion["n"] >= 100].sort_values("adjusted")
y = [f"{r} (n={n})" for r, n in byRegion["n"].items()]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hlines(y, byRegion["adjusted"] / 1000, byRegion["paycheck"] / 1000, color="#c3c2b7", linewidth=2)
ax.scatter(byRegion["paycheck"] / 1000, y, color="#86b6ef", s=64, zorder=3, label="Paycheck")
ax.scatter(byRegion["adjusted"] / 1000, y, color="#2a78d6", s=64, zorder=3, label="Adjusted for cost of living")
for yi, v in zip(y, byRegion["adjusted"]):
    ax.annotate(f"${v / 1000:.1f}k", (v / 1000, yi), xytext=(0, 9), textcoords="offset points", ha="center",
                color="#52514e", fontsize=9)
ax.margins(x=0.1, y=0.15)
ax.set(xlabel="Median first-job salary ($k)", title="DC pays more on paper; after cost of living, remote pays most and Baltimore beats DC")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Chart: median raise when changing jobs, by type of move (job history), lollipop (moves up highlighted)
# raise = new job's salary vs the same person's previous job
jobs = jobs.sort_values(["campus_id", "start_date"])
raisePct = (jobs["annual_salary_usd"] / jobs.groupby("campus_id")["annual_salary_usd"].shift() - 1) * 100
byMove = raisePct.groupby(jobs["change_type"]).agg(["median", "size"]).dropna().sort_values("median")
up = ["Internal Promotion", "New Employer - Advance"]
y = [f"{m} (n={n})" for m, n in byMove["size"].items()]
colors = ["#2a78d6" if m in up else "#c3c2b7" for m in byMove.index]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hlines(y, 0, byMove["median"], color=colors, linewidth=2)
ax.scatter(byMove["median"], y, color=colors, s=64, zorder=3)
for yi, v in zip(y, byMove["median"]):
    ax.annotate(f"+{v:.0f}%", (v, yi), xytext=(8, 0), textcoords="offset points", va="center", color="#52514e")
ax.set(xlim=(0, byMove["median"].max() * 1.2), xlabel="Median raise vs previous job (%)",
       title="Move up, not sideways: step-up moves pay about 20%, lateral ones about 4%")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Chart: seniority of jobs started N years after the first job (job history), 100% stacked columns
# Lead, Manager and Director are grouped as "Lead+"; light to dark = junior to senior
firstStart = jobs[jobs["change_type"] == "First Job"].set_index("campus_id")["start_date"]
yearsIn = ((jobs["start_date"] - jobs["campus_id"].map(firstStart)).dt.days / 365.25).round().clip(upper=8).astype(int)
level = jobs["seniority_level"].replace({"Lead": "Lead+", "Manager": "Lead+", "Director": "Lead+"})
ladder = pd.crosstab(yearsIn, level, normalize="index")[["Entry", "Mid", "Senior", "Lead+"]] * 100
ramp = {"Entry": "#cde2fb", "Mid": "#86b6ef", "Senior": "#2a78d6", "Lead+": "#0d366b"}

fig, ax = plt.subplots(figsize=(9, 4.5))
bottom = np.zeros(len(ladder))
xlabels = [f"{y}+" if y == 8 else str(y) for y in ladder.index]
for name, color in ramp.items():
    ax.bar(xlabels, ladder[name], bottom=bottom, color=color, width=0.7, edgecolor="white", linewidth=1.5, label=name)
    bottom += ladder[name]
seniorPlus = ladder["Senior"] + ladder["Lead+"]
ax.set(ylim=(0, 100), xlabel="Years after the first job started", ylabel="% of jobs started that year",
       title=f"Five years in, {seniorPlus.loc[5]:.0f}% of new roles are Senior or higher")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1), frameon=False, reverse=True)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Table: statistical checks behind each chart title
# chi-square for rates, Mann-Whitney for skewed money/time gaps; the four Baltimore comparisons are Bonferroni-corrected
mw = lambda x, y: stats.mannwhitneyu(x.dropna(), y.dropna()).pvalue
responded = stuAlum[stuAlum["first_destination"] != "No Response"]
fullTime = responded["first_destination"] == "Employed Full-Time"
viaOffer = hired["first_job_found_via"] == "Return Offer from Internship"
pay = {r: g for r, g in firstJobs.groupby("region")}
upMove = jobs["change_type"].isin(["Internal Promotion", "New Employer - Advance"])
lateral = jobs["change_type"].isin(["Internal Lateral", "New Employer - Lateral"])
tests = [
    ("2020 grads were less often employed full-time", "Chi-square",
     stats.chi2_contingency(pd.crosstab(responded["graduation_year"] == 2020, fullTime)).pvalue),
    ("Return offers start faster than other routes", "Mann-Whitney",
     mw(hired["months_to_first_job"][viaOffer], hired["months_to_first_job"][~viaOffer])),
    ("DC paychecks are higher than Baltimore's", "Mann-Whitney",
     mw(pay["Washington, DC Metro"]["annual_salary_usd"], pay["Baltimore, MD"]["annual_salary_usd"])),
    ("After cost of living, remote beats Baltimore", "Mann-Whitney x4",
     4 * mw(pay["Remote - United States"]["adjusted"], pay["Baltimore, MD"]["adjusted"])),
] + [(f"After cost of living, Baltimore beats {r}", "Mann-Whitney x4", 4 * mw(pay["Baltimore, MD"]["adjusted"], pay[r]["adjusted"]))
     for r in ["Washington, DC Metro", "Northern Virginia", "Columbia/Howard County, MD"]] + [
    ("Step-up moves raise pay more than lateral ones", "Mann-Whitney", mw(raisePct[upMove], raisePct[lateral])),
]
pd.DataFrame(tests, columns=["claim", "test", "p"]).assign(
    p=lambda d: d["p"].clip(upper=1), verdict=lambda d: np.where(d["p"] < 0.05, "supported", "not significant")).assign(
    p=lambda d: d["p"].map(lambda p: f"{p:.1e}"))